# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [4]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [5]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [6]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [15]:
# 1
by_city = df.groupby('city').agg(orders=('order_id', 'count'), cancellations=('cancel', 'mean'), delivery_days=('delivery_days', 'mean')).round(3)
by_city["revenue"] = paid.groupby("city")["amount"].sum()
srt = by_city.sort_values("revenue")
print(srt)
print("Алматы выделяется по количеству заказов и по выручке")

           orders  cancellations  delivery_days     revenue
city                                                       
Атырау        143          0.077          2.036   1893850.0
Актобе        387          0.106          2.070   5117380.0
Караганда     519          0.091          1.941   7020200.0
Шымкент       776          0.206          4.451   8602660.0
Астана       1339          0.088          2.019  17884750.0
Алматы       2116          0.078          1.997  29144090.0
Алматы выделяется по количеству заказов и по выручке


## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [31]:
# 2

p = paid.pivot_table(index = "category", columns = "month", values = "amount", aggfunc = ["sum", "mean"] ).round(3)
print(p)
print("---------")
print(((p[('sum', 10)] - p[('sum', 9)]) / p[('sum', 9)] * 100).round(2))
print(len(paid))

                    sum                   mean           
month                9           10         9          10
category                                                 
Дом           4997730.0   5125790.0  10499.433  10768.466
Книги         1065300.0   1140910.0   4020.000   3815.753
Красота       2610950.0   2678990.0   6187.085   6158.598
Одежда        9167100.0   8505730.0  13520.796  13045.598
Электроника  17402330.0  16968100.0  32772.750  33600.198
---------
category
Дом            2.56
Книги          7.10
Красота        2.61
Одежда        -7.21
Электроника   -2.50
dtype: float64
4739


## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [37]:
# 3
gr = df.groupby("delivery_days").agg(cancellations=("cancel", "mean"), orders=("order_id", "count")).round(3)
print(gr.loc[8.0, "cancellations"]/gr.loc[1.0, "cancellations"].round(2))
print("Долгие сроки доставки стали ключевым фактором всплеска отмен и просадки в Шымкенте")

10.825
Долгие сроки доставки стали ключевым фактором всплеска отмен и просадки в Шымкенте


## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [ ]:
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
sh_pct = p.loc["Шымкент", "chg"] * 100
print(f"Выручка в Шымкенте изменилась на {sh_pct:.1f}%")
m_paid = m[m[status] == "оплачено"]]

city
Актобе      -0.014
Алматы       0.010
Астана      -0.020
Атырау       0.050
Караганда   -0.076
Шымкент     -0.113
Name: chg, dtype: float64
Выручка в шымкенте упала на 11.3%


## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [58]:
# 5
print(len(df))
print(len(cu))
m = df.merge(cu, how = "left", on = "customer_id")
print(len(m), m["customer_id"].isna().sum())
m["age_group"] = m["age"] // 10 * 10

print(m.groupby("age_group").agg(number_of_checks = ("order_id", "count"), cancellation = ("cancel", "mean")).round(3))
print(f"\n{m.loc[m['age_group'] == 60]['order_id'].count()} заказов от клиентов 60+ лет")
print(len(sorted(m.loc[m["age_group"] == 60, "age"].unique())), "возрастов в группе 60+")

m_paid = m[m["status"] == "оплачен"]
print(m_paid.groupby("age_group")["amount"].agg(["mean", "median"]).round(3)) 

print("Нет, медиана не подтверждает вывод AI. Средний чек в группе 60 действительно самый высокий (16 779 ₸), но это вызвано единичными дорогими покупками (выбросами). Медианный чек группы 60 составляет 10 495 ₸ — это меньше, чем у групп 10 и 30 лет, и находится на обычном уровне всех остальных покупателей.")



5280
2200
5280 0
           number_of_checks  cancellation
age_group                                
10                      229         0.087
20                     1226         0.096
30                     1241         0.111
40                     1170         0.111
50                     1294         0.092
60                      120         0.133

120 заказов от клиентов 60+ лет
1 возрастов в группе 60+
                mean   median
age_group                    
10         14044.737  10700.0
20         14130.415  10100.0
30         15142.339  10670.0
40         14792.971  10050.0
50         14671.796  10240.0
60         16779.135  10495.0
Нет, медиана не подтверждает вывод AI. Средний чек в группе 60 действительно самый высокий (16 779 ₸), но это вызвано единичными дорогими покупками (выбросами). Медианный чек группы 60 составляет 10 495 ₸ — это меньше, чем у групп 10 и 30 лет, и находится на обычном уровне всех остальных покупателей.


## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [85]:
# 6
overall = m.groupby("customer_id").size().sum()
print(m.groupby("customer_id").size().value_counts().sort_index())
orders_per_cust =m.groupby("customer_id").size()
print(f"Доля клиентов, сделавших более одного заказа: {((orders_per_cust > 1).sum()/ overall):.1%}")
mon = m.groupby("customer_id")["month"].nunique()
print(f"Клиенты, которые делали заказы в два месяца: {(mon == 2).sum()}") 
print(f"{(~cu['customer_id'].isin(orders_per_cust.index)).sum()} клиентов не сделали ни одного заказа") 


1     501
2     563
3     434
4     293
5     127
6      58
7      15
8       9
9       1
10      1
Name: count, dtype: int64
Доля клиентов, сделавших более одного заказа: 28.4%
Клиенты, которые делали заказы в два месяца: 1064
198 клиентов не сделали ни одного заказа


In [81]:
cu

,customer_id,channel,age,signup_date
0,C00001,рефералы,31,2026-08-06
1,C00002,реклама,30,2025-07-16
2,C00003,органика,45,2026-05-22
3,C00004,органика,43,2025-10-11
4,C00005,реклама,35,2025-09-02
...,...,...,...,...
2195,C02196,органика,51,2025-07-05
2196,C02197,органика,55,2026-04-22
2197,C02198,органика,57,2025-12-24
2198,C02199,реклама,60,2025-04-18


## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [ ]:
# 7

## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [ ]:
# 8